# EV purchases (S6E9) — pipeline v2

Changes vs v1 (`fe_ensemble_pipeline.py`, LB 0.94618):

1. **Original-dataset features** (`ev_adoption.csv`): per income / commute value → count in original, lift, novel flag, distance to nearest original value, frequency rank
2. **More digit features** (commute digits, round-number flags)
3. **Sharper target encoding**: income/commute at `m=3` with bins 100/500/2000 and 1/5 km; all columns **+ digits** at `m=10` and `m=100`
4. **Shallow, column-subsampled trees** (depth 5, 30% of columns per tree) for LightGBM, XGBoost, CatBoost
5. **10 folds** instead of 5
6. **Hill-climbing blend** over v2 models **and** your saved v1 predictions
7. **Hard edges**: income ≥ 170,537 → always Yes; 31,004–41,970 → always No (100% / 0% in train)

**Tip:** for a quick check first, set `N_FOLDS = 5` and `MODELS = ["lgb"]` and compare with v1 LightGBM OOF (0.9459).

In [1]:
import os, time, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier

## Config

In [2]:
TRAIN_PATH, TEST_PATH, ORIG_PATH = "train_cleaned.csv", "test.csv", "ev_adoption.csv"
OUT_DIR, TAG = "preds", "v2"          # predictions saved as preds/oof_v2_<model>.npy
TARGET = "Will_Buy_EV"
SEED, N_FOLDS, INNER_FOLDS = 42, 10, 5
MODELS = ["lgb", "xgb", "cat"]        # add "mlp" for extra diversity (slow)
LR = 0.03                             # learning rate for LightGBM / XGBoost
BASELINE_OOF = 0.94605                # v1 blend OOF

CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
            "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]
INC, COM = "Annual_Income_USD", "Daily_Commute_km"
os.makedirs(OUT_DIR, exist_ok=True)

## Load data

In [3]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
for df in (train, test, orig):
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip()

y = (train.pop(TARGET) == "Yes").astype(int).to_numpy()
test_ids = test["id"].to_numpy()
full = pd.concat([train.drop(columns="id"), test.drop(columns="id")], ignore_index=True)
RAW_COLS = list(full.columns)
ntr, nte = len(train), len(test)
tr_rows, te_rows = np.arange(ntr), np.arange(ntr, ntr + nte)

X_base = full.copy()
for c in CAT_COLS:
    X_base[c] = pd.factorize(X_base[c])[0].astype(np.int8)

inc = full[INC].round().astype(np.int64).to_numpy()          # income in dollars
com = (full[COM] * 10).round().astype(np.int64).to_numpy()   # commute in 0.1 km
print(f"train {ntr} | test {nte} | original {len(orig)} | pos rate {y.mean():.4f}")

train 668665 | test 286571 | original 10000 | pos rate 0.1746


## 1. Digit features
Synthetic-generator artifacts live in the digits of income and commute.

In [4]:
for i in range(6):                                   # income: ones ... hundred-thousands
    X_base[f"inc_d{i}"] = ((inc // 10**i) % 10).astype(np.int8)
X_base["com_d_dec"] = (com % 10).astype(np.int8)          # 0.1 km digit
X_base["com_d1"]    = ((com // 10) % 10).astype(np.int8)  # 1 km digit
X_base["inc_mod100"]  = (inc % 100 == 0).astype(np.int8)
X_base["inc_mod1000"] = (inc % 1000 == 0).astype(np.int8)
X_base["inc_is_30k"]  = (inc == 30000).astype(np.int8)
X_base["com_is_int"]  = (com % 10 == 0).astype(np.int8)
X_base["com_is_5"]    = (com == 50).astype(np.int8)
DIGIT_COLS = ["inc_d1", "inc_d2", "inc_d3", "inc_d4", "com_d_dec", "com_d1"]

## 2. Value statistics vs. the original dataset
For every exact income / commute value (target-free, so computed on train+test):
- `cnt` — how often it appears in train+test
- `cnt_orig` — how often it appears in the original 10k rows
- `lift` — share here ÷ share in original (over-produced values have high lift)
- `novel` — value never appears in the original
- `dist_orig` — signed distance to the nearest original value
- `freqrank` — rank of `cnt` among all values

In [5]:
def value_stats(keys, orig_keys, scale, prefix):
    u, inv, cnt = np.unique(keys, return_inverse=True, return_counts=True)
    uo, cnto = np.unique(orig_keys, return_counts=True)
    pos = np.clip(np.searchsorted(uo, u), 0, len(uo) - 1)
    found = uo[pos] == u
    cnt_orig = np.where(found, cnto[pos], 0)
    share_all, share_orig = cnt / len(keys), cnt_orig / len(orig_keys)
    lift = share_all / np.where(share_orig > 0, share_orig, 1 / len(orig_keys))
    # signed distance to nearest original value
    p = np.searchsorted(uo, u)
    lo = uo[np.clip(p - 1, 0, len(uo) - 1)]
    hi = uo[np.clip(p, 0, len(uo) - 1)]
    d_lo, d_hi = (u - lo) / scale, (hi - u) / scale
    dist = np.where(np.abs(d_lo) <= np.abs(d_hi), -d_lo, d_hi)
    dist[found] = 0.0
    rank = cnt.argsort().argsort() / len(cnt)
    return pd.DataFrame({
        f"{prefix}_cnt": cnt[inv].astype(np.int32),
        f"{prefix}_cnt_orig": cnt_orig[inv].astype(np.int16),
        f"{prefix}_lift": np.log1p(lift[inv]).astype(np.float32),
        f"{prefix}_novel": (~found)[inv].astype(np.int8),
        f"{prefix}_dist_orig": dist[inv].astype(np.float32),
        f"{prefix}_freqrank": rank[inv].astype(np.float32),
    })

o_inc = orig[INC].dropna().round().astype(np.int64).to_numpy()
o_com = (orig[COM].dropna() * 10).round().astype(np.int64).to_numpy()
X_base = pd.concat([X_base,
                    value_stats(inc, o_inc, 1, "inc"),
                    value_stats(com, o_com, 10, "com")], axis=1)
print("base features:", X_base.shape[1])
X_base.filter(like="inc_").describe().T.head(12)

base features: 38


,count,mean,std,min,25%,50%,75%,max
inc_d0,955236.0,4.002961,3.066168,0.000000,1.000000,4.000000,7.000000,9.000000
inc_d1,955236.0,4.255856,2.973261,0.000000,1.000000,4.000000,7.000000,9.000000
inc_d2,955236.0,5.033014,2.988900,0.000000,3.000000,5.000000,8.000000,9.000000
inc_d3,955236.0,3.789998,2.851212,0.000000,1.000000,3.000000,6.000000,9.000000
inc_d4,955236.0,5.311078,2.963458,0.000000,3.000000,6.000000,8.000000,9.000000
inc_d5,955236.0,0.273277,0.445642,0.000000,0.000000,0.000000,1.000000,1.000000
inc_mod100,955236.0,0.100752,0.301000,0.000000,0.000000,0.000000,0.000000,1.000000
inc_mod1000,955236.0,0.092101,0.289168,0.000000,0.000000,0.000000,0.000000,1.000000
inc_is_30k,955236.0,0.091999,0.289025,0.000000,0.000000,0.000000,0.000000,1.000000
inc_cnt,955236.0,8306.395741,25330.257998,1.000000,117.000000,201.000000,343.000000,87881.000000


## 3 & 4. Target-encoding specs
`name -> (integer codes over train+test, number of codes, smoothing m)`

- **Income / commute** at `m=3` (sharp): exact value + several bin widths
- **Every other column + digit columns + income/1000**, each at `m=10` and `m=100`

In [6]:
te_specs = {}
def add_spec(name, key, m):
    codes, uniq = pd.factorize(key)
    te_specs[name] = (codes.astype(np.int64), len(uniq), m)

# income / commute, sharp smoothing
add_spec("te_inc", inc, 3)
for w in [100, 500, 2000]:
    add_spec(f"te_inc_b{w}", inc // w, 3)
add_spec("te_com", com, 3)
for w in [10, 50]:                          # 1 km, 5 km  (com is in 0.1 km)
    add_spec(f"te_com_b{w}", com // w, 3)

# all other columns + digits, two smoothing levels
OTHER = [c for c in RAW_COLS if c not in (INC, COM)]
for m in (10, 100):
    for c in OTHER + DIGIT_COLS:
        add_spec(f"te_{c}_m{m}", X_base[c].to_numpy(), m)
    add_spec(f"te_inc_b1000_m{m}", inc // 1000, m)
print("target encodings:", len(te_specs))

target encodings: 43


In [7]:
def te_map(codes_fit, y_fit, codes_apply, n, m):
    """Smoothed target mean per code: (sum_y + prior*m) / (count + m)."""
    prior = y_fit.mean()
    s = np.bincount(codes_fit, weights=y_fit, minlength=n)
    c = np.bincount(codes_fit, minlength=n)
    return ((s + prior * m) / (c + m))[codes_apply].astype(np.float32)

def build_te(fit_idx, apply_idx_list):
    """Nested TE: fit rows get inner-fold encodings; apply rows use all fit rows."""
    y_fit = y[fit_idx]
    inner = list(KFold(INNER_FOLDS, shuffle=True, random_state=SEED).split(fit_idx))
    fit_out, apply_out = {}, [{} for _ in apply_idx_list]
    for name, (codes, n, m) in te_specs.items():
        c_fit = codes[fit_idx]
        col = np.empty(len(fit_idx), dtype=np.float32)
        for a, b in inner:
            col[b] = te_map(c_fit[a], y_fit[a], c_fit[b], n, m)
        fit_out[name] = col
        for k, idx in enumerate(apply_idx_list):
            apply_out[k][name] = te_map(c_fit, y_fit, codes[idx], n, m)
    return pd.DataFrame(fit_out), [pd.DataFrame(d) for d in apply_out]

def make_xy(fit_idx, apply_idx_list):
    te_fit, te_apply = build_te(fit_idx, apply_idx_list)
    X_fit = pd.concat([X_base.iloc[fit_idx].reset_index(drop=True), te_fit], axis=1)
    X_apply = [pd.concat([X_base.iloc[i].reset_index(drop=True), t], axis=1)
               for i, t in zip(apply_idx_list, te_apply)]
    return X_fit, X_apply

## 5. Models — shallow, column-subsampled trees
Depth 5 and only 30% of columns per tree: with ~90 overlapping features, this forces
each tree to use different views of the same signal.

In [8]:
def fit_lgb(Xtr, ytr, Xva, yva):
    m = lgb.LGBMClassifier(
        objective="binary", n_estimators=20000, learning_rate=LR,
        max_depth=5, num_leaves=32, min_child_samples=10,
        colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
        random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
          callbacks=[lgb.early_stopping(300, verbose=False)])
    return m, m.best_iteration_

def fit_xgb(Xtr, ytr, Xva, yva):
    m = xgb.XGBClassifier(
        n_estimators=20000, learning_rate=LR, tree_method="hist",
        max_depth=5, min_child_weight=10, colsample_bytree=0.3, subsample=0.8,
        reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
        eval_metric="auc", early_stopping_rounds=300,
        random_state=SEED, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m, m.best_iteration

def fit_cat(Xtr, ytr, Xva, yva):
    m = CatBoostClassifier(
        loss_function="Logloss", eval_metric="AUC", iterations=20000,
        learning_rate=0.08, depth=5, rsm=0.3, l2_leaf_reg=2.0, border_count=1024,
        early_stopping_rounds=300, random_seed=SEED, verbose=0, thread_count=-1)
    m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
    return m, m.get_best_iteration()

def fit_mlp(Xtr, ytr, Xva, yva):
    num_cols = [c for c in Xtr.columns if c not in CAT_COLS]
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                                    subsample=200_000, random_state=SEED), num_cols)])
    m = make_pipeline(pre, MLPClassifier(
        hidden_layer_sizes=(256, 128), alpha=1e-4, batch_size=2048,
        learning_rate_init=1e-3, max_iter=40, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=5, random_state=SEED))
    m.fit(Xtr, ytr)
    return m, m[-1].n_iter_

FITTERS = {"lgb": fit_lgb, "xgb": fit_xgb, "cat": fit_cat, "mlp": fit_mlp}

## 6. Cross-validation loop
Per fold: build encodings from that fold's training rows only → train → predict validation (OOF) and test.
Test prediction = average of the fold models.

In [9]:
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof = {k: np.zeros(ntr) for k in MODELS}
tst = {k: np.zeros(nte) for k in MODELS}
fold_auc = {k: [] for k in MODELS}

for f, (tr, va) in enumerate(skf.split(tr_rows, y), 1):
    t0 = time.time()
    Xtr, (Xva, Xte) = make_xy(tr, [va, te_rows])
    print(f"\nfold {f}/{N_FOLDS}: {Xtr.shape[1]} features ({time.time()-t0:.0f}s)")
    for k in MODELS:
        t1 = time.time()
        model, it = FITTERS[k](Xtr, y[tr], Xva, y[va])
        oof[k][va] = model.predict_proba(Xva)[:, 1]
        tst[k] += model.predict_proba(Xte)[:, 1] / N_FOLDS
        fold_auc[k].append(roc_auc_score(y[va], oof[k][va]))
        print(f"  {k}: AUC={fold_auc[k][-1]:.5f}  iters={it}  ({time.time()-t1:.0f}s)", flush=True)

for k in MODELS:
    np.save(f"{OUT_DIR}/oof_{TAG}_{k}.npy", oof[k]); np.save(f"{OUT_DIR}/test_{TAG}_{k}.npy", tst[k])
    s = roc_auc_score(y, oof[k])
    print(f"{k}: mean fold {np.mean(fold_auc[k]):.5f} +- {np.std(fold_auc[k]):.5f} | OOF {s:.5f}")


fold 1/10: 81 features (6s)
  lgb: AUC=0.94513  iters=951  (84s)
  xgb: AUC=0.94509  iters=743  (146s)
  cat: AUC=0.94527  iters=542  (165s)

fold 2/10: 81 features (7s)
  lgb: AUC=0.94535  iters=798  (82s)
  xgb: AUC=0.94535  iters=1032  (272s)
  cat: AUC=0.94542  iters=911  (338s)

fold 3/10: 81 features (11s)
  lgb: AUC=0.94569  iters=1000  (135s)
  xgb: AUC=0.94566  iters=1107  (300s)
  cat: AUC=0.94562  iters=790  (272s)

fold 4/10: 81 features (9s)
  lgb: AUC=0.94611  iters=775  (119s)
  xgb: AUC=0.94614  iters=1098  (260s)
  cat: AUC=0.94617  iters=894  (311s)

fold 5/10: 81 features (15s)
  lgb: AUC=0.94696  iters=810  (104s)
  xgb: AUC=0.94691  iters=1336  (294s)
  cat: AUC=0.94692  iters=548  (222s)

fold 6/10: 81 features (9s)
  lgb: AUC=0.94730  iters=762  (123s)
  xgb: AUC=0.94726  iters=961  (243s)
  cat: AUC=0.94724  iters=547  (204s)

fold 7/10: 81 features (9s)
  lgb: AUC=0.94611  iters=581  (88s)
  xgb: AUC=0.94606  iters=558  (145s)
  cat: AUC=0.94611  iters=433  (1

## 7. Hill-climbing blend over all saved predictions (v1 + v2)
Uses every `preds/oof_*.npy` that has a matching `test_*.npy`. Ranks are blended (AUC only cares about order).

In [10]:
def to_rank(p):
    return rankdata(p) / len(p)

def hill_climb(oofs, y, max_steps=200):
    names = list(oofs)
    P = np.column_stack([to_rank(oofs[n]) for n in names])
    counts, blend, best = np.zeros(len(names)), np.zeros(len(y)), -1.0
    for step in range(max_steps):
        cand = [(blend * step + P[:, j]) / (step + 1) for j in range(len(names))]
        scores = [roc_auc_score(y, c) for c in cand]
        j = int(np.argmax(scores))
        if scores[j] <= best + 1e-7:
            break
        best, blend = scores[j], cand[j]
        counts[j] += 1
    return {n: float(w) for n, w in zip(names, counts / counts.sum()) if w > 0}, best

all_oof, all_tst = {}, {}
for f in sorted(glob.glob(f"{OUT_DIR}/oof_*.npy")):
    name = os.path.basename(f)[4:-4]
    t = f"{OUT_DIR}/test_{name}.npy"
    if os.path.exists(t):
        all_oof[name], all_tst[name] = np.load(f), np.load(t)
        print(f"{name:12s} OOF {roc_auc_score(y, all_oof[name]):.5f}")

weights, blend_auc = hill_climb(all_oof, y)
print("\nweights:", {k: round(v, 3) for k, v in weights.items()})
print(f"blend OOF {blend_auc:.5f} | vs v1 blend {blend_auc - BASELINE_OOF:+.5f}")

cat          OOF 0.94600
lgb          OOF 0.94590
mlp          OOF 0.94362
v2_cat       OOF 0.94623
v2_lgb       OOF 0.94621
v2_xgb       OOF 0.94619

weights: {'cat': 0.2, 'v2_cat': 0.4, 'v2_lgb': 0.2, 'v2_xgb': 0.2}
blend OOF 0.94630 | vs v1 blend +0.00025


## 8. Hard edges + submission
In train, income ≥ 170,537 is always Yes and 31,004–41,970 is always No.

In [11]:
def hard_edges(pred, income):
    p = pred.copy()
    p[income >= 170_537] = 1.0
    p[(income >= 31_004) & (income <= 41_970)] = 0.0
    return p

oof_blend = sum(w * to_rank(all_oof[k]) for k, w in weights.items())
tst_blend = sum(w * to_rank(all_tst[k]) for k, w in weights.items())
inc_tr, inc_te = inc[:ntr], inc[ntr:]
print(f"blend OOF {roc_auc_score(y, oof_blend):.6f} -> with hard edges "
      f"{roc_auc_score(y, hard_edges(oof_blend, inc_tr)):.6f}")

sub = pd.DataFrame({"id": test_ids, TARGET: hard_edges(tst_blend, inc_te)})
sub.to_csv(f"submission_{TAG}_blend.csv", index=False)
for k in MODELS:
    pd.DataFrame({"id": test_ids, TARGET: hard_edges(tst[k], inc_te)}).to_csv(f"submission_{TAG}_{k}.csv", index=False)
print(f"saved submission_{TAG}_blend.csv (+ per-model files)")
sub.head()

blend OOF 0.946304 -> with hard edges 0.946311
saved submission_v2_blend.csv (+ per-model files)


,id,Will_Buy_EV
0,668665,0.575700
1,668666,0.492790
2,668667,0.296024
3,668668,0.231934
4,668669,0.560618
